In [2]:
import pandas as pd

jpm = pd.read_csv("data/features_JPM.csv", parse_dates=["Date"])
jpm = jpm.set_index("Date").sort_index()

y = jpm["target_return_h1"]
feature_columns = [
    column for column in jpm.columns
    if column != "close" and not column.startswith("target_")
]
X = jpm[feature_columns]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Duplicate dates:", jpm.index.duplicated().sum())
print("Missing next-day returns:", y.isna().sum())

X shape: (2791, 50)
y shape: (2791,)
Duplicate dates: 0
Missing next-day returns: 1


In [3]:
print("Last date in the dataset:", y.index[-1].date())
print("Dates with a missing next-day return:", y.index[y.isna()].date)

Last date in the dataset: 2026-11-20
Dates with a missing next-day return: [datetime.date(2026, 11, 20)]


In [4]:
has_target = y.notna()
X_model = X.loc[has_target]
y_model = y.loc[has_target]

print("Predictor table shape:", X_model.shape)
print("Target shape:", y_model.shape)
print("Dates match:", X_model.index.equals(y_model.index))
print("Missing predictor values:", X_model.isna().sum().sum())

Predictor table shape: (2790, 50)
Target shape: (2790,)
Dates match: True
Missing predictor values: 0


In [5]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ridge_pipeline = make_pipeline(StandardScaler(), Ridge())
ridge_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gbm_model = GradientBoostingRegressor(random_state=42)
gbm_model

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf

In [7]:
import json
from pathlib import Path

feature_groups = json.loads(
    Path("data/feature_groups_JPM.json").read_text(encoding="utf-8")
)

print("Top-level entries:", list(feature_groups))

Top-level entries: ['ticker', 'download_date', 'seed', 'first_date', 'last_date', 'main_target', 'targets', 'groups', 'all_features']


In [8]:
ablation_groups = feature_groups["groups"]
expected_features = set(feature_groups["all_features"])
model_features = set(X_model.columns)

print("Feature groups:", list(ablation_groups))
print("Missing from X_model:", sorted(expected_features - model_features))
print("Extra in X_model:", sorted(model_features - expected_features))

Feature groups: ['returns', 'volatility', 'technical', 'volume', 'other_markets', 'sector', 'calendar']
Missing from X_model: []
Extra in X_model: []


In [9]:
for group_name, columns in ablation_groups.items():
    missing_columns = sorted(set(columns) - model_features)
    print(f"{group_name}: {len(columns)} predictors | missing: {missing_columns}")

returns: 9 predictors | missing: []
volatility: 6 predictors | missing: []
technical: 9 predictors | missing: []
volume: 1 predictors | missing: []
other_markets: 15 predictors | missing: []
sector: 3 predictors | missing: []
calendar: 7 predictors | missing: []


In [10]:
ridge_param_grid = {
    "ridge__alpha": [0.01, 0.1, 1, 10, 100]
}
gbm_param_grid = {
    "max_depth": [1, 2],
    "learning_rate": [0.05, 0.1],
}

In [11]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit


def fit_tuned_ridge(X_train, y_train):
    search = GridSearchCV(
        estimator=ridge_pipeline,
        param_grid=ridge_param_grid,
        cv=TimeSeriesSplit(n_splits=3, gap=1),
        scoring="neg_mean_squared_error",
    )
    return search.fit(X_train, y_train)

In [12]:
def fit_tuned_gbm(X_train, y_train):
    search = GridSearchCV(
        estimator=gbm_model,
        param_grid=gbm_param_grid,
        cv=TimeSeriesSplit(n_splits=3, gap=1),
        scoring="neg_mean_squared_error",
    )
    return search.fit(X_train, y_train)

In [13]:
from group_utils import get_splits, check_splits

splits = list(get_splits(len(X_model), horizon=1))
check_splits(splits, X_model.index)

train_idx, test_idx = splits[0]

print("Number of splits:", len(splits))
print("First split — training ends:", X_model.index[train_idx[-1]].date())
print("First split — testing starts:", X_model.index[test_idx[0]].date())
print("Total test rows:", sum(len(test) for _, test in splits))

Number of splits: 29
First split — training ends: 2019-10-07
First split — testing starts: 2019-10-08
Total test rows: 1790


In [14]:
def walk_forward_predictions(X, y, splits, fit_model):
    predictions = []

    for train_idx, test_idx in splits:
        fitted = fit_model(X.iloc[train_idx], y.iloc[train_idx])
        X_test = X.iloc[test_idx]

        predictions.append(
            pd.Series(fitted.predict(X_test), index=X_test.index)
        )

    return pd.concat(predictions)


ridge_pred = walk_forward_predictions(X_model, y_model, splits, fit_tuned_ridge)

print("Ridge predictions:", len(ridge_pred))
print("First prediction date:", ridge_pred.index[0].date())
print("Duplicate prediction dates:", ridge_pred.index.duplicated().sum())

Ridge predictions: 1790
First prediction date: 2019-10-08
Duplicate prediction dates: 0


In [15]:
gbm_pred = walk_forward_predictions(X_model, y_model, splits, fit_tuned_gbm)

print("Gradient Boosting predictions:", len(gbm_pred))
print("Test dates match Ridge:", gbm_pred.index.equals(ridge_pred.index))
print("Missing predictions:", gbm_pred.isna().sum())

Gradient Boosting predictions: 1790
Test dates match Ridge: True
Missing predictions: 0


In [16]:
gbm_param_grid = {
    "max_depth": [1, 2],
    "learning_rate": [0.05, 0.1],
}

In [17]:
from group_utils import save_results

y_test = y_model.loc[ridge_pred.index]

save_results("ridge", y_test, ridge_pred)
save_results("gbm", y_test, gbm_pred)

Saved 1790 predictions to C:\Users\Ndimanya\Desktop\B.I2.Lab\TmeSeries\results\ridge.csv
Saved 1790 predictions to C:\Users\Ndimanya\Desktop\B.I2.Lab\TmeSeries\results\gbm.csv


WindowsPath('C:/Users/Ndimanya/Desktop/B.I2.Lab/TmeSeries/results/gbm.csv')

In [18]:
from sklearn.metrics import root_mean_squared_error

full_rmse = root_mean_squared_error(y_test, ridge_pred)
ablation_predictions = {}
ablation_rows = []

for group_name, columns in ablation_groups.items():
    print(f"Testing Ridge without {group_name}...", flush=True)

    X_reduced = X_model.drop(columns=columns)
    reduced_pred = walk_forward_predictions(
        X_reduced, y_model, splits, fit_tuned_ridge
    )

    assert reduced_pred.index.equals(y_test.index)
    ablation_predictions[group_name] = reduced_pred

    reduced_rmse = root_mean_squared_error(y_test, reduced_pred)
    ablation_rows.append({
        "Group removed": group_name,
        "RMSE without group": reduced_rmse,
        "RMSE change": reduced_rmse - full_rmse,
    })

print(f"Full Ridge RMSE: {full_rmse:.8f}")
ablation_table = pd.DataFrame(ablation_rows)
display(ablation_table.round(8))

Testing Ridge without returns...
Testing Ridge without volatility...
Testing Ridge without technical...
Testing Ridge without volume...
Testing Ridge without other_markets...
Testing Ridge without sector...
Testing Ridge without calendar...
Full Ridge RMSE: 0.01961985


,Group removed,RMSE without group,RMSE change
0,returns,0.019564,-5.590000e-05
1,volatility,0.019555,-6.463000e-05
2,technical,0.019554,-6.619000e-05
3,volume,0.019619,-3.600000e-07
4,other_markets,0.019251,-3.693200e-04
5,sector,0.019617,-3.120000e-06
6,calendar,0.019616,-3.890000e-06


In [19]:
print(ablation_table.round(8).to_string(index=False))

Group removed  RMSE without group   RMSE change
      returns            0.019564 -5.590000e-05
   volatility            0.019555 -6.463000e-05
    technical            0.019554 -6.619000e-05
       volume            0.019619 -3.600000e-07
other_markets            0.019251 -3.693200e-04
       sector            0.019617 -3.120000e-06
     calendar            0.019616 -3.890000e-06


In [20]:
import inspect
from group_utils import dm_vs_naive

print(inspect.getsource(dm_vs_naive))

def dm_vs_naive(y_true, y_pred, horizon=1):
    """
    Diebold-Mariano test of the model against the naive (zero) forecast,
    squared-error loss, Harvey-Leybourne-Newbold correction (tslab version).

    Negative statistic = the model has the smaller errors.
    p < 0.05 = the difference is unlikely to be luck.
    Pass horizon=h for h-day targets (Mark M.'s multi-horizon table).
    """
    y_true = np.asarray(y_true, float)
    stat, p = diebold_mariano(y_true, np.asarray(y_pred, float), np.zeros(len(y_true)), horizon)
    return stat, p, dm_verdict(stat, p)



In [21]:
from group_utils import diebold_mariano

dm_results = {}

for group_name, reduced_pred in ablation_predictions.items():
    assert y_test.index.equals(reduced_pred.index)
    assert y_test.index.equals(ridge_pred.index)

    stat, p_value = diebold_mariano(
        y_test.to_numpy(),
        reduced_pred.to_numpy(),
        ridge_pred.to_numpy(),
        1,
    )
    dm_results[group_name] = (stat, p_value)

ablation_table["DM statistic"] = ablation_table["Group removed"].map(
    lambda name: dm_results[name][0]
)
ablation_table["DM p-value"] = ablation_table["Group removed"].map(
    lambda name: dm_results[name][1]
)

print(ablation_table.round(8).to_string(index=False))

Group removed  RMSE without group   RMSE change  DM statistic  DM p-value
      returns            0.019564 -5.590000e-05     -1.007049    0.314047
   volatility            0.019555 -6.463000e-05     -0.931329    0.351809
    technical            0.019554 -6.619000e-05     -1.820954    0.068781
       volume            0.019619 -3.600000e-07     -0.046736    0.962729
other_markets            0.019251 -3.693200e-04     -3.149848    0.001661
       sector            0.019617 -3.120000e-06     -0.212666    0.831612
     calendar            0.019616 -3.890000e-06     -0.167241    0.867199


### Feature ablation conclusion

Removing `other_markets` gave the largest improvement in Ridge’s next-day JPM return forecasts. On the same 1,790 test dates, RMSE fell from 0.019620 to 0.019251, a reduction of about 1.88%. The Diebold–Mariano test showed a significant difference compared with full Ridge (p = 0.001661). The other six removals showed no statistically significant difference at the 5% level.

In [22]:
from group_utils import dm_vs_naive, naive_forecast, score

naive_pred = naive_forecast(y_test.index)
print("Naïve metrics:", score(y_test, naive_pred))

for name, predictions in [("Ridge", ridge_pred), ("Gradient Boosting", gbm_pred)]:
    print(f"\n{name} metrics:", score(y_test, predictions))
    stat, p_value, verdict = dm_vs_naive(y_test, predictions, horizon=1)
    print(f"DM statistic: {stat:.6f} | p-value: {p_value:.6f}")
    print(verdict)

Naïve metrics: {'RMSE (return)': 0.01889776729370046, 'MAE (return)': 0.012514073495143765, 'R2 out-of-sample': 0.0, 'Directional accuracy': nan}

Ridge metrics: {'RMSE (return)': 0.019619851030750573, 'MAE (return)': 0.012984354752919175, 'R2 out-of-sample': -0.07788000945733731, 'Directional accuracy': 0.5}
DM statistic: 2.712172 | p-value: 0.006748
Significantly WORSE than naive

Gradient Boosting metrics: {'RMSE (return)': 0.019713127332916084, 'MAE (return)': 0.01262374186444874, 'R2 out-of-sample': -0.08815324297987615, 'Directional accuracy': 0.5256983240223464}
DM statistic: 1.585187 | p-value: 0.113101
No significant difference


### Model comparison conclusion

The naïve zero-return forecast had the lowest test RMSE (0.018898), followed by Ridge (0.019620) and Gradient Boosting (0.019713). The Diebold–Mariano test showed that Ridge performed significantly worse than the naïve forecast (p = 0.006748). For Gradient Boosting, the test did not detect a significant difference (p = 0.113101). Neither model improved on the naïve forecast in this evaluation.